# ml_C_trees_case.ipynb

**所属章节**：Chapter C 树模型  
**主题**：树模型在金融因子预测中的应用——对比 Lasso、随机森林、XGBoost，并用 SHAP 解释哪些因子最重要  
**数据**：模拟 A 股因子数据（n=500，p=20 个财务因子，时序结构）  
**运行说明**：顺序执行，约 2–3 分钟。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.linear_model import LassoCV, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap, warnings, os
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

FP  = fm.FontProperties(fname='/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
FPB = fm.FontProperties(fname='/usr/share/fonts/opentype/noto/NotoSansCJK-Medium.ttc')
C = {'primary':'#2166AC','secondary':'#D6604D','tertiary':'#4DAC26',
     'neutral':'#878787','highlight':'#B2182B','fill':'#D1E5F0'}
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'grid.linestyle':'--'})
SEED=42; np.random.seed(SEED)
print('设置完成')

---
## 1. 数据生成（模拟 A 股月度因子数据）

模拟 500 只股票的月度超额收益率，20 个财务因子（PE倒数、ROE、动量等），
收益率与若干因子存在非线性关系（体现树模型优势）。


In [ ]:
# 模拟因子数据（n=500个月度观测，p=20个财务因子）
n, p = 500, 20
factor_names = ['EP','ROE','ROA','BM','MOM_1M','MOM_3M','MOM_6M',
                'SIZE','LEV','TURNOVER','GP','OP','INV','AGE',
                'CASH','VOL','BETA','ILLIQ','STDRET','SKEW']

# 因子矩阵（模拟相关结构）
rho = 0.3
Sig = rho ** np.abs(np.subtract.outer(np.arange(p), np.arange(p)))
X_raw = np.random.multivariate_normal(np.zeros(p), Sig, n)

# 非线性收益率生成（前5个因子有效，包含非线性交互）
# 注意：引入非线性项，体现树模型相对线性模型的优势
y = (0.8*X_raw[:,0]                      # EP：线性正向
   + 0.6*X_raw[:,1]                      # ROE：线性正向
   - 0.5*np.abs(X_raw[:,4])              # MOM_1M：非线性
   + 0.4*X_raw[:,2]*X_raw[:,3]           # ROA x BM 交互项
   + 0.3*np.sin(X_raw[:,5])              # MOM_3M：周期性非线性
   + np.random.normal(0, 1.5, n))        # 噪声

# 按时间排序（模拟时序结构）
df = pd.DataFrame(X_raw, columns=factor_names)
df['ret'] = y
df['date'] = pd.date_range('2000-01', periods=n, freq='ME')
df = df.sort_values('date').reset_index(drop=True)

print(f'数据维度：{df.shape}')
print(f'时间范围：{df.date.min().date()} 至 {df.date.max().date()}')
print(f'收益率统计：均值={df.ret.mean():.3f}，标准差={df.ret.std():.3f}')

---
## 2. 时序分割与数据准备


In [ ]:
# 时序分割：前80%训练，后20%测试（不随机，防止前瞻偏差）
split = int(n * 0.8)
X_tr_raw = df[factor_names].values[:split]
X_te_raw = df[factor_names].values[split:]
y_train  = df['ret'].values[:split]
y_test   = df['ret'].values[split:]

# 线性模型需要标准化，树模型不需要
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr_raw)
X_te_sc = sc.transform(X_te_raw)

# 树模型直接使用原始数据
X_train = X_tr_raw
X_test  = X_te_raw

print(f'训练集：{X_train.shape[0]} 个样本（{df.date.iloc[:split].dt.year.min()}–{df.date.iloc[:split].dt.year.max()}）')
print(f'测试集：{X_test.shape[0]} 个样本（{df.date.iloc[split:].dt.year.min()}–{df.date.iloc[split:].dt.year.max()}）')

---
## 3. 五种方法的估计


In [ ]:
# 样本外 R2 辅助函数
def oos_r2(y_true, y_pred, baseline_mean):
    return 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-baseline_mean)**2)

bm = y_train.mean()  # 基准：训练集均值

# 方法一：Lasso（线性，标准化数据）
lasso = LassoCV(cv=5, n_alphas=50, random_state=SEED, max_iter=3000)
lasso.fit(X_tr_sc, y_train)
yp_lasso = lasso.predict(X_te_sc)
n_lasso  = (lasso.coef_ != 0).sum()

# 方法二：Ridge（线性，标准化数据）
from sklearn.linear_model import RidgeCV
ridge = RidgeCV(alphas=np.logspace(-2,4,30), cv=5)
ridge.fit(X_tr_sc, y_train)
yp_ridge = ridge.predict(X_te_sc)

# 方法三：随机森林（树模型，原始数据，开启 OOB）
rf = RandomForestRegressor(n_estimators=300, oob_score=True,
                            random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
yp_rf = rf.predict(X_test)

# 方法四：XGBoost（树模型，原始数据）
xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, verbosity=0)
xgb_model.fit(X_train, y_train)
yp_xgb = xgb_model.predict(X_test)

# 汇总结果
results = {}
for name, yp in [('Lasso', yp_lasso),('Ridge', yp_ridge),
                  ('随机森林', yp_rf),('XGBoost', yp_xgb)]:
    results[name] = {
        'MSE'   : round(mean_squared_error(y_test,yp), 4),
        'OOS_R2': round(oos_r2(y_test, yp, bm), 4)
    }
df_res = pd.DataFrame(results).T
print(df_res.to_string())
print(f'\n随机森林 OOB R² = {rf.oob_score_:.4f}')
print(f'Lasso 选中变量数 = {n_lasso}/{p}')

---
## 4. 结果可视化：预测对比


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.subplots_adjust(wspace=0.35)

# 左：OOS R2 对比条形图
ax = axes[0]
methods = df_res.index.tolist()
r2s = df_res['OOS_R2'].values
cols_b = [C['tertiary'], C['fill'], C['primary'], C['secondary']]
bars = ax.barh(methods, r2s, color=cols_b, alpha=0.85, height=0.5)
ax.axvline(0,'k--',lw=0.8,alpha=0.5)
for bar,val in zip(bars,r2s):
    ax.text(val+0.003, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', fontproperties=FP, fontsize=9, va='center')
ax.set_xlabel('样本外 R²', fontproperties=FP, fontsize=11)
ax.set_title('五种方法样本外预测对比', fontproperties=FPB, fontsize=12)
for t in ax.get_yticklabels(): t.set_fontproperties(FP)

# 右：随机森林 vs Lasso 预测散点图
ax2 = axes[1]
ax2.scatter(y_test, yp_lasso, color=C['tertiary'], alpha=0.5, s=18, label='Lasso')
ax2.scatter(y_test, yp_rf,    color=C['primary'],  alpha=0.5, s=18, label='随机森林')
lim = max(abs(y_test).max(),abs(yp_rf).max())*1.05
ax2.plot([-lim,lim],[-lim,lim],'k--',lw=1,alpha=0.4)
ax2.set_xlabel('真实值', fontproperties=FP, fontsize=11)
ax2.set_ylabel('预测值', fontproperties=FP, fontsize=11)
ax2.set_title('预测值 vs 真实值', fontproperties=FPB, fontsize=12)
ax2.legend(prop=FP)

fig.suptitle('非线性因子数据上的方法对比（树模型优势场景）',
             fontproperties=FPB, fontsize=13)
fig.savefig('figs/fig_C_case_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## 5. SHAP 分析：哪些财务因子最重要？


In [ ]:
# 计算随机森林的 SHAP 值
explainer_rf = shap.TreeExplainer(rf)
shap_vals    = explainer_rf.shap_values(X_test)

# 全局特征重要性（|SHAP| 均值）
shap_importance = pd.Series(
    np.abs(shap_vals).mean(0), index=factor_names
).sort_values(ascending=False)
print('SHAP 特征重要性（前10）：')
print(shap_importance.head(10).round(4).to_string())

# beeswarm 图
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals, X_test, feature_names=factor_names,
                  max_display=10, show=False)
plt.title('A股因子 SHAP Beeswarm 图（随机森林）',
          fontproperties=FPB, fontsize=12)
plt.tight_layout()
plt.savefig('figs/fig_C_case_shap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('\nSHAP beeswarm 图已保存')

---
## 提示词模板 #4b：SHAP 可解释性分析

```
背景：已训练随机森林 rf_model，对其做 SHAP 可解释性分析。

我的数据：X_test（测试集），feature_names（特征名列表）

请帮我：
1. 用 shap.TreeExplainer 计算 SHAP 值
2. 绘制 beeswarm 图（前10个特征，max_display=10）
3. 对测试集第0个样本绘制 waterfall 图
4. 打印每个特征的平均 |SHAP| 值排名（前10）
5. 所有图的标题和轴标签用中文，代码用中文注释
```


In [ ]:
print('='*55)
print('Chapter C 案例分析完成')
print('='*55)
best = df_res['OOS_R2'].idxmax()
print(f'最优方法: {best}，OOS R² = {df_res.loc[best,"OOS_R2"]:.4f}')
print()
print('输出文件:')
print('  figs/fig_C_case_comparison.png')
print('  figs/fig_C_case_shap.png')